# CPG Evaluation Notebook

This notebook demonstrates the full evaluation pipeline:
1. Load the test passage
2. Generate paraphrases using CPG and LLM
3. Compute all metrics
4. Visualize results
5. Error analysis

In [ ]:
import sys
sys.path.insert(0, '..')

from src.model import CustomParaphraseGenerator
from src.evaluate import evaluate_paraphrase
from src.utils import load_test_sample, preprocess_text, validate_input
import json

## 1. Load Test Passage

In [ ]:
test_text = preprocess_text(load_test_sample())
validation = validate_input(test_text)
print(f"Word count: {validation['word_count']}")
print(f"Valid: {validation['valid']}")
print(f"\nFirst 200 chars: {test_text[:200]}...")

## 2. Generate CPG Paraphrase

In [ ]:
# Use base model or fine-tuned checkpoint
# For fine-tuned: cpg = CustomParaphraseGenerator(model_path='./cpg-finetuned-final')
cpg = CustomParaphraseGenerator()
cpg_result = cpg.paraphrase(test_text)

print(f"Input words: {cpg_result['input_words']}")
print(f"Output words: {cpg_result['output_words']}")
print(f"Length ratio: {cpg_result['length_ratio']}")
print(f"Latency: {cpg_result['latency_ms']}ms")
print(f"Sentences: {cpg_result['num_sentences']}")
print(f"\nParaphrased text:\n{cpg_result['paraphrased_text']}")

## 3. Generate LLM Paraphrase

In [ ]:
try:
    from src.llm_baseline import get_available_generator
    llm = get_available_generator()
    llm_result = llm.paraphrase(test_text)
    
    print(f"Model: {llm_result['model_name']}")
    print(f"Output words: {llm_result['output_words']}")
    print(f"Length ratio: {llm_result['length_ratio']}")
    print(f"Latency: {llm_result['latency_ms']}ms")
    print(f"\nParaphrased text:\n{llm_result['paraphrased_text']}")
except RuntimeError as e:
    print(f"LLM unavailable: {e}")
    llm_result = None

## 4. Evaluate Both

In [ ]:
import pandas as pd

cpg_metrics = evaluate_paraphrase(test_text, cpg_result['paraphrased_text'])
cpg_metrics['latency_ms'] = cpg_result['latency_ms']

data = {'Metric': list(cpg_metrics.keys()), 'CPG': list(cpg_metrics.values())}

if llm_result:
    llm_metrics = evaluate_paraphrase(test_text, llm_result['paraphrased_text'])
    llm_metrics['latency_ms'] = llm_result['latency_ms']
    data['LLM'] = list(llm_metrics.values())

df = pd.DataFrame(data)
print(df.to_string(index=False))

## 5. Visualize

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

quality = ['bleu', 'rouge1_f1', 'rouge2_f1', 'rougeL_f1', 'bert_score_f1']
labels = ['BLEU', 'ROUGE-1', 'ROUGE-2', 'ROUGE-L', 'BERTScore']

cpg_vals = [cpg_metrics[m] for m in quality]
cpg_vals[0] /= 100  # normalize BLEU

x = np.arange(len(labels))
fig, ax = plt.subplots(figsize=(10, 6))
w = 0.35
ax.bar(x - w/2, cpg_vals, w, label='CPG', color='#2196F3')

if llm_result:
    llm_vals = [llm_metrics[m] for m in quality]
    llm_vals[0] /= 100
    ax.bar(x + w/2, llm_vals, w, label='LLM', color='#FF9800')

ax.set_ylabel('Score')
ax.set_title('Quality Metrics Comparison')
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Error Analysis

Let's look at sentence-level differences to understand what changed.

In [ ]:
import nltk

input_sents = nltk.sent_tokenize(test_text)
cpg_sents = nltk.sent_tokenize(cpg_result['paraphrased_text'])

print(f"Input sentences: {len(input_sents)}")
print(f"CPG output sentences: {len(cpg_sents)}")
print()

for i, (inp, out) in enumerate(zip(input_sents, cpg_sents)):
    from src.evaluate import compute_jaccard
    j = compute_jaccard(inp, out)
    print(f"--- Sentence {i+1} (Jaccard={j:.2f}) ---")
    print(f"IN:  {inp[:100]}...")
    print(f"OUT: {out[:100]}...")
    print()